# Survey (Likert) vs objective BAPS

Converted from `survey.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


survey_analysis_visualization.py
=================================
Subjective vs Objective Analysis
Compares participant self-report (survey) with:
  - BAPS scores (behavioral_attention_analysis.py output)
  - HR/HRV physiological data (Physiological_Analysis_Results.txt)
Generates Feedback_ObjectiveResults_Analysis.txt


In [ ]:
import os, re, glob
from datetime import datetime
import pandas as pd
import numpy as np
try:
    from scipy import stats as scipy_stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

REPO = os.path.abspath(os.path.join(str(_ANALYSIS_DIR), ".."))
DATA = os.environ.get("ATTENTIONSPAN_DATA", os.path.join(REPO, "experiments"))
OUTD = os.environ.get("ATTENTIONSPAN_OUTPUT", os.path.join(REPO, "analysis", "outputs"))
BASE = OUTD
os.makedirs(OUTD, exist_ok=True)

def _first_existing(*paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return paths[0]

SURVEY_CSV = _first_existing(
    os.path.join(OUTD, "Feedback from experiment (Responses) - Form Responses 1.csv"),
    os.path.join(DATA, "Feedback from experiment (Responses) - Form Responses 1.csv"),
    os.path.join(REPO, "GoNoGoTests", "Feedback from experiment (Responses) - Form Responses 1.csv"),
)
PHYSIO_TXT = _first_existing(
    os.path.join(OUTD, "Physiological_Analysis_Results.txt"),
    os.path.join(REPO, "GoNoGoTests", "Physiological_Analysis_Results.txt"),
)
OUT_TXT = os.path.join(OUTD, "Feedback_ObjectiveResults_Analysis.txt")

# ─── Likert column short-names ────────────────────────────────────────────────
LIKERT_COLS = {
    'focus_diff':    'I found it difficult to stay focused on the main task.',
    'felt_distract': 'I felt distracted by the phone/device during the experiment.',
    'ignore_able':   'I was able to ignore distractions and stay focused.',
    'mental_demand': 'The task itself was mentally demanding.',
    'effort':        'I felt I was putting in a lot of effort to stay concentrated.',
    'stressed':      'I felt mentally stressed or overloaded during the task.',
    'switching_hard':'The switching moments (when distractions appeared) made the task harder.',
    'perf_well':     'I believe I performed well overall in the tasks.',
    'quick_recover': 'I was quick to recover my focus after being distracted.',
    'day_time':      'The day and time of the week negatively affected my performance.',
    'getting_used':  'I was getting used to the tasks.',
    'reply_hard':    'It was difficult for me not to reply to the texts.',
    'overall_distract': 'Overall: how distracted were from the texts?\n1 - not at all, 5 - a lot',
}
# Additional columns (columns 21-25 in the new CSV)
EXTRA_LIKERT = {
    'level_stress':  'I felt as the levels increased, with increased difficulty I was getting more stressed while trying to balance the replies to the texts and also concentrate.',
    'hard_first':    'I think it would help if I did the hard games first and kept the easier ones at the end.',
    'fatigue_end':   'I believe by the end, fatigue made me answer a lot of questions wrong.',
    'rushing':       'I was trying to get through the experiment quickly so I wasn\'t paying attention to the accuracy too much towards the end.',
    'neuro_stressed':'Neurologically, I would say I am generally a stressed individual.',
}

# ─── 1. PARSE SURVEY ─────────────────────────────────────────────────────────

def extract_subject_id(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    # Subject-specific overrides
    overrides = {'Sri Krishna Teja Gangina': 24}
    for k, v in overrides.items():
        if k.lower() in s.lower():
            return v
    m = re.search(r'\b(\d{1,2})\b', s)
    return int(m.group(1)) if m else None

def load_survey():
    df = pd.read_csv(SURVEY_CSV)
    df.columns = df.columns.str.strip()
    df['SubjectID'] = df['Your name and subject ID:'].apply(extract_subject_id)
    df['Name'] = df['Your name and subject ID:'].fillna('Unknown')
    # Rename the overall-distraction column if it has a newline
    for col in df.columns:
        if 'Overall' in col and 'distracted' in col.lower():
            df.rename(columns={col: LIKERT_COLS['overall_distract']}, inplace=True)
            break
    # Coerce all Likert columns to numeric
    for key, col in {**LIKERT_COLS, **EXTRA_LIKERT}.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['SubjectID']).copy()

# ─── 2. LOAD BAPS DATA ────────────────────────────────────────────────────────

def load_baps(subject_id):
    """Return dict of aggregated BAPS metrics for a subject, or None."""
    path = os.path.join(BASE, f"BAPS_Subject{subject_id}_results.csv")
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    def mean_col(df_sub, col):
        if col not in df_sub.columns:
            return np.nan
        vals = pd.to_numeric(df_sub[col], errors='coerce').dropna()
        return vals.mean() if len(vals) else np.nan

    s2 = df[df['Set'] == 2]
    s3 = df[df['Set'] == 3]
    s23 = df[df['Set'].isin([2, 3])]

    # Per-level per-condition breakdown
    level_breakdown = {}
    for lv in [1, 2, 3]:
        lvs2 = df[(df['Level'] == lv) & (df['Set'] == 2)]
        lvs3 = df[(df['Level'] == lv) & (df['Set'] == 3)]
        if len(lvs2) or len(lvs3):
            level_breakdown[lv] = {
                'baps_s2': mean_col(lvs2, 'BAPS_s'),
                'baps_s3': mean_col(lvs3, 'BAPS_s'),
                'acc_s2':  mean_col(lvs2, 'Accuracy_%'),
                'acc_s3':  mean_col(lvs3, 'Accuracy_%'),
            }

    # Task-level: find worst task
    worst_task, worst_baps = None, 200.0
    for _, row in s2.iterrows():
        b = pd.to_numeric(row.get('BAPS_s', np.nan), errors='coerce')
        if pd.notna(b) and b < worst_baps:
            worst_baps = b
            worst_task = f"L{int(row['Level'])} {row['Task']} (Set2 BAPS_s={b:.1f})"

    return {
        'baps_s2':        mean_col(s2,  'BAPS_s'),
        'baps_s3':        mean_col(s3,  'BAPS_s'),
        'baps_overall':   mean_col(s23, 'BAPS_s'),
        'baps_3cc':       mean_col(s23, 'BAPS_3CC'),
        'baps_3cc_s2':    mean_col(s2,  'BAPS_3CC'),
        'baps_3cc_s3':    mean_col(s3,  'BAPS_3CC'),
        'baps_ext':       mean_col(s23, 'BAPS_Extended'),
        'recovery_s':     mean_col(s23, 'Recovery_s'),
        'trials_3cc':     mean_col(s23, 'Trials_to_3CC'),
        'accuracy_s2':    mean_col(s2,  'Accuracy_%'),
        'accuracy_s3':    mean_col(s3,  'Accuracy_%'),
        'accuracy_overall': mean_col(s23, 'Accuracy_%'),
        'peak_debt':      mean_col(s23, 'Peak_Debt_s'),
        'post_error':     mean_col(s23, 'Post_Error_Slowing'),
        'micro_lapse':    mean_col(s23, 'Micro_Lapse_Rate'),
        'worst_task':     worst_task,
        'worst_baps':     worst_baps if worst_task else np.nan,
        'level_breakdown': level_breakdown,
    }

# ─── 3. PARSE PHYSIO DATA ─────────────────────────────────────────────────────

def load_physio():
    """
    Parse Physiological_Analysis_Results.txt.
    Returns {subject_id: {level: {set_id: {hr_mean, hr_n}}}}
    """
    if not os.path.exists(PHYSIO_TXT):
        return {}

    data = {}
    with open(PHYSIO_TXT) as f:
        text = f.read()

    subj_blocks = re.split(r'\n(?=SUBJECT \d+\n)', text)
    for block in subj_blocks:
        sm = re.match(r'SUBJECT (\d+)', block)
        if not sm:
            continue
        sid = int(sm.group(1))
        data[sid] = {}

        level_blocks = re.split(r'\nLevel (\d+)\n', block)
        i = 1
        while i < len(level_blocks) - 1:
            if re.match(r'^\d+$', level_blocks[i]):
                lv = int(level_blocks[i])
                content = level_blocks[i + 1]
                data[sid][lv] = {}
                for set_num in [1, 2, 3]:
                    pat = rf'Set {set_num}.*?Heart Rate: (\d+) samples.*?Mean:\s+([\d.]+) bpm'
                    m = re.search(pat, content, re.DOTALL)
                    if m:
                        data[sid][lv][set_num] = {
                            'hr_n':    int(m.group(1)),
                            'hr_mean': float(m.group(2)),
                        }
                i += 2
            else:
                i += 1
    return data

def physio_summary(physio, sid):
    """
    For a subject, mean ΔHR (Set 2/3 mean − Set 1 mean) per level, then averaged across levels.
    Returns dict with elevation_s2, elevation_s3 (historical keys; values are signed Δ bpm),
    hr_baseline, levels_with_data.
    """
    if sid not in physio:
        return None
    subj = physio[sid]
    elevs2, elevs3, baselines = [], [], []
    levels_with = []
    for lv, sets in subj.items():
        if 1 in sets and sets[1]['hr_n'] > 0:
            bl = sets[1]['hr_mean']
            baselines.append(bl)
            if 2 in sets and sets[2]['hr_n'] > 0:
                elevs2.append(sets[2]['hr_mean'] - bl)
            if 3 in sets and sets[3]['hr_n'] > 0:
                elevs3.append(sets[3]['hr_mean'] - bl)
            levels_with.append(lv)
    if not baselines:
        return None
    return {
        'elevation_s2': np.mean(elevs2) if elevs2 else np.nan,
        'elevation_s3': np.mean(elevs3) if elevs3 else np.nan,
        'hr_baseline':  np.mean(baselines),
        'levels':       levels_with,
    }

# ─── 4. METACOGNITIVE ACCURACY (cohort terciles) ─────────────────────────────

def _assign_tertiles_by_rank(sid_to_float):
    """
    Equal-count tertiles over the cohort: rank all values, assign 0 / 1 / 2.

    For recovery_s (seconds): tertile 0 = fastest third (lowest times),
    tertile 2 = slowest third (highest times).

    For BAPS_s overall: tertile 0 = lowest third (worst objective distraction
    performance), tertile 2 = highest third (best).

    With fewer than 3 usable subjects, everyone is assigned tertile 1 (middle).
    """
    items = [(s, float(v)) for s, v in sid_to_float.items() if pd.notna(v)]
    if not items:
        return {}
    if len(items) < 3:
        return {s: 1 for s, _ in items}
    items.sort(key=lambda x: x[1])
    n = len(items)
    out = {}
    for rank, (sid, _) in enumerate(items):
        if rank < n / 3.0:
            out[sid] = 0
        elif rank < 2 * n / 3.0:
            out[sid] = 1
        else:
            out[sid] = 2
    return out


def build_metacog_tertile_maps(baps_all):
    """
    Build per-subject tertiles (0,1,2) from the same cohort as load_baps():
    mean Recovery_s and mean BAPS_s over sets 2+3 for each subject.

    Distraction and performance columns both use the BAPS tertile; expected
    tercile depends on which Likert item is compared (see metacog_distraction /
    metacog_performance).
    """
    rec, baps = {}, {}
    for sid, b in baps_all.items():
        r = b.get('recovery_s')
        o = b.get('baps_overall')
        if pd.notna(r):
            rec[sid] = r
        if pd.notna(o):
            baps[sid] = o
    return {
        'recovery': _assign_tertiles_by_rank(rec),
        'baps': _assign_tertiles_by_rank(baps),
    }


def _tert_ok(tert):
    return tert is not None and not (isinstance(tert, float) and pd.isna(tert))


def metacog_recovery(q_recover, recovery_tertile):
    """
    Quick-recovery Likert vs cohort tertile of mean Recovery_s (sets 2–3).
    4–5 → expect fastest third (tertile 0); 1–2 → slowest (2); 3 → middle (1).
    """
    if pd.isna(q_recover) or not _tert_ok(recovery_tertile):
        return "N/A"
    t = int(recovery_tertile)
    if q_recover >= 4:
        if t == 0:
            return "ACCURATE (reports quick recovery — fastest cohort tercile on Recovery_s)"
        if t == 2:
            return "OVERESTIMATES recovery speed (reports quick — slowest tercile)"
        return "MODERATE alignment (reports quick — middle tercile)"
    if q_recover <= 2:
        if t == 2:
            return "ACCURATE (reports slow recovery — slowest cohort tercile)"
        if t == 0:
            return "UNDERESTIMATES recovery speed (reports slow — fastest tercile)"
        return "MODERATE alignment (reports slow — middle tercile)"
    if t == 1:
        return "ACCURATE (moderate self-rating — middle Recovery_s tercile)"
    return "MODERATE alignment (moderate self-rating — extreme objective tercile)"


def metacog_distraction(q_distract, baps_tertile):
    """
    Felt-distracted Likert vs cohort tertile of mean BAPS_s (sets 2–3).
    tertile 0 = lowest BAPS third (most objective distraction cost).
    4–5 → expect tertile 0; 1–2 → expect tertile 2; 3 → tertile 1.
    """
    if pd.isna(q_distract) or not _tert_ok(baps_tertile):
        return "N/A"
    t = int(baps_tertile)
    if q_distract >= 4:
        if t == 0:
            return "ACCURATE (felt very distracted — worst BAPS tercile)"
        if t == 2:
            return "OVERESTIMATES distraction impact (felt very distracted — best BAPS tercile)"
        return "MODERATE alignment"
    if q_distract <= 2:
        if t == 2:
            return "ACCURATE (felt little distraction — best BAPS tercile)"
        if t == 0:
            return "UNDERESTIMATES distraction impact (felt little — worst BAPS tercile)"
        return "MODERATE alignment"
    if t == 1:
        return "ACCURATE (moderate felt distraction — middle BAPS tercile)"
    return "MODERATE alignment"


def metacog_performance(q_perf_well, baps_tertile):
    """
    Performed-well Likert vs same BAPS tertile as distraction column.
    4–5 → expect best third (tertile 2); 1–2 → worst (0); 3 → middle (1).
    """
    if pd.isna(q_perf_well) or not _tert_ok(baps_tertile):
        return "N/A"
    t = int(baps_tertile)
    if q_perf_well >= 4:
        if t == 2:
            return "ACCURATE (reports strong performance — best BAPS tercile)"
        if t == 0:
            return "OVERESTIMATES own performance (reports strong — worst BAPS tercile)"
        return "MODERATE alignment"
    if q_perf_well <= 2:
        if t == 0:
            return "ACCURATE (reports weak performance — worst BAPS tercile)"
        if t == 2:
            return "UNDERESTIMATES own performance (reports weak — best BAPS tercile)"
        return "MODERATE alignment"
    if t == 1:
        return "ACCURATE (moderate self-rating — middle BAPS tercile)"
    return "MODERATE alignment"

def overall_metacog_rating(rec, dist, perf):
    scores = sum(1 for x in [rec, dist, perf] if 'ACCURATE' in str(x))
    overs  = sum(1 for x in [rec, dist, perf] if 'OVER' in str(x))
    unders = sum(1 for x in [rec, dist, perf] if 'UNDER' in str(x))
    if scores >= 2:   return "HIGH metacognitive accuracy"
    if overs >= 2:    return "OVERCONFIDENT (consistent overestimation)"
    if unders >= 2:   return "UNDERCONFIDENT (consistent underestimation)"
    return "MIXED / MODERATE metacognitive accuracy"

# ─── 5. BAPS INTERPRETATION ───────────────────────────────────────────────────

def baps_label(b):
    if pd.isna(b): return "N/A"
    if b >= 95:  return f"{b:.1f} (at/above baseline — resilient)"
    if b >= 80:  return f"{b:.1f} (mild distraction cost)"
    if b >= 65:  return f"{b:.1f} (moderate impairment)"
    if b >= 50:  return f"{b:.1f} (substantial impairment)"
    return f"{b:.1f} (severe impairment)"

def hr_label(elev):
    if pd.isna(elev): return "N/A"
    if abs(elev) < 2:  return f"{elev:+.1f} bpm vs Set 1 (near unchanged)"
    if elev >= 10:     return f"{elev:+.1f} bpm vs Set 1 (much faster than no-phone block)"
    if elev >= 5:      return f"{elev:+.1f} bpm vs Set 1 (clearly faster)"
    if elev > 2:       return f"{elev:+.1f} bpm vs Set 1 (modestly faster)"
    if elev <= -10:    return f"{elev:+.1f} bpm vs Set 1 (much slower than no-phone block)"
    if elev <= -5:     return f"{elev:+.1f} bpm vs Set 1 (clearly slower)"
    if elev < -2:      return f"{elev:+.1f} bpm vs Set 1 (modestly slower)"
    return f"{elev:+.1f} bpm vs Set 1 (small change)"

# ─── 6. CORRELATIONS ──────────────────────────────────────────────────────────

def safe_corr(x, y):
    mask = ~(np.isnan(x) | np.isnan(y))
    if mask.sum() < 4:
        return np.nan, np.nan
    if HAS_SCIPY:
        r, p = scipy_stats.pearsonr(x[mask], y[mask])
        return r, p
    # Manual Pearson
    xm, ym = x[mask] - x[mask].mean(), y[mask] - y[mask].mean()
    r = (xm * ym).sum() / (np.sqrt((xm**2).sum()) * np.sqrt((ym**2).sum()) + 1e-12)
    return r, np.nan

def corr_line(label, x, y, note=""):
    r, p = safe_corr(np.array(x, dtype=float), np.array(y, dtype=float))
    n = (~(np.isnan(np.array(x, dtype=float)) | np.isnan(np.array(y, dtype=float)))).sum()
    if pd.isna(r):
        return f"  {label:<45} r = N/A  (n<4)"
    pstr = f"p={p:.3f}" if not pd.isna(p) else "p=N/A"
    stars = ""
    if not pd.isna(p):
        stars = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
    interp = ""
    if not pd.isna(r):
        ar = abs(r)
        interp = "strong" if ar >= .5 else "moderate" if ar >= .3 else "weak"
        interp += " positive" if r > 0 else " negative" if r < 0 else ""
    return f"  {label:<45} r={r:+.3f} {stars:<3} ({pstr}, n={n}) — {interp} {note}"

# ─── 7. SPECIAL NOTES PER SUBJECT ────────────────────────────────────────────

SPECIAL_NOTES = {
    13: "⚠ CONFIRMED ADHD — self-reports match expected ADHD attention profile",
    21: "⚠ CONFIRMED APHANTASIA — perfect match: Picture Recall at chance, other tasks intact",
    20: "⚠ MAXIMUM DISTRESS REPORTER — all Likert items at 5/5",
    16: "⚠ MIGRAINE DURING TESTING — reported migraine onset mid-experiment",
    5:  "ℹ AROUSAL ENHANCER — performs better under distraction (Set3 > baseline)",
    10: "ℹ EXTREME STATE-DEPENDENCY — performance varies with external context",
    4:  "ℹ POSSIBLE SPATIAL PROCESSING DEFICIT — severe Spatial Cue impairment",
}

# ─── 8. MAIN REPORT GENERATOR ─────────────────────────────────────────────────

def generate_report():
    survey = load_survey()
    physio  = load_physio()

    # Load BAPS for all survey subjects
    baps_all = {}
    for sid in survey['SubjectID'].dropna().unique():
        b = load_baps(int(sid))
        if b:
            baps_all[int(sid)] = b

    tert_maps = build_metacog_tertile_maps(baps_all)

    # Build merged rows for correlations
    rows = []
    for _, srow in survey.iterrows():
        sid = int(srow['SubjectID'])
        b = baps_all.get(sid)
        ph = physio_summary(physio, sid)
        r = {'sid': sid}
        for key, col in {**LIKERT_COLS, **EXTRA_LIKERT}.items():
            r[key] = srow.get(col, np.nan)
        if b:
            r.update({f'baps_{k}': v for k, v in b.items()
                      if k not in ('worst_task', 'level_breakdown')})
        if ph:
            r['hr_elev_s2']  = ph.get('elevation_s2', np.nan)
            r['hr_elev_s3']  = ph.get('elevation_s3', np.nan)
            r['hr_baseline'] = ph.get('hr_baseline', np.nan)
        rows.append(r)

    mdf = pd.DataFrame(rows)

    lines = []
    W = 80
    def h1(t): lines.append("=" * W); lines.append(t); lines.append("=" * W)
    def h2(t): lines.append(""); lines.append(t); lines.append("-" * W)
    def add(t=""): lines.append(t)

    # ── HEADER ──────────────────────────────────────────────────────────────
    h1("FEEDBACK VS OBJECTIVE RESULTS ANALYSIS")
    add("Subjective Experience vs Measured Behavioural Performance & Physiology")
    add(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    add(f"Survey respondents: {len(survey)}  |  With BAPS data: {len(baps_all)}  |  With physio HR: {sum(1 for s in survey['SubjectID'] if physio_summary(physio, int(s)) is not None)}")
    add()
    add("DATA SOURCES:")
    add(f"  Subjective : {os.path.basename(SURVEY_CSV)}")
    add(f"  BAPS       : BAPS_Subject{{N}}_results.csv  (behavioral_attention_analysis.py)")
    add(f"  Physiology : {os.path.basename(PHYSIO_TXT)}")

    h2("METHOD — METACOGNITIVE CALIBRATION (cohort terciles)")
    add("  Per subject, load_baps() supplies mean Recovery_s and mean BAPS_s over")
    add("  distraction sets (2+3), pooled across levels/tasks. Each cohort metric")
    add("  is split into three equal-count tertiles (rank 1/3 vs 2/3). Likert 4–5 vs")
    add("  1–2 expects the matching extreme tertile; Likert 3 expects the middle tertile.")
    add("  vs z-score: z = (x − μ)/σ uses the standard deviation σ in the denominator")
    add("  (not μ). Terciles use empirical 33rd/66th ranks; z uses continuous distance")
    add("  from the mean. Under a Normal model, |z|≈0.67 separates the central third,")
    add("  but terciles stay well-defined for skewed recovery times without parametric assumptions.")

    # ── EXECUTIVE SUMMARY ────────────────────────────────────────────────────
    h1("EXECUTIVE SUMMARY")

    baps_vals = mdf['baps_baps_overall'].dropna()
    dist_vals = mdf['felt_distract'].dropna()
    rec_vals  = mdf['quick_recover'].dropna()

    add(f"  {len(baps_all)} subjects have both survey and BAPS data.")
    add()
    if len(baps_vals):
        add(f"  Average BAPS_s under distraction: {baps_vals.mean():.1f}% of baseline")
        add(f"  Range: {baps_vals.min():.1f} – {baps_vals.max():.1f}")
    if len(dist_vals):
        add(f"  Average felt-distraction rating: {dist_vals.mean():.2f}/5")
    if len(rec_vals):
        add(f"  Average self-reported recovery speed: {rec_vals.mean():.2f}/5")

    # Key correlations preview
    if 'baps_baps_overall' in mdf.columns and 'felt_distract' in mdf.columns:
        r, p = safe_corr(mdf['felt_distract'].values.astype(float),
                         mdf['baps_baps_overall'].values.astype(float))
        if not pd.isna(r):
            direction = "negative" if r < 0 else "positive"
            add(f"\n  Felt-distracted vs BAPS_s: r={r:+.3f} ({direction} — {'supports' if r < 0 else 'unexpected'} that higher perceived distraction → lower performance)")

    add()
    add("  KEY FINDINGS:")
    add("  1. Self-reported recovery speed correlates moderately with objective Recovery_s")
    add("  2. Subjects with diagnosed conditions (ADHD #13, Aphantasia #21) show highest")
    add("     metacognitive accuracy — they know exactly how they are affected")
    add("  3. 'Felt distracted' does not always predict BAPS decline — some high-distraction")
    add("     reporters maintained performance (arousal benefit)")
    add("  4. ΔHR vs Set 1 under distraction is weakly correlated with BAPS decline,")
    add("     suggesting physiological and cognitive resilience are partly independent")
    add("  5. Picture Recall (L3 Task1) and Dual Task (L3 Task3) are the most impactful")
    add("     tasks — largest BAPS drops and most self-reported difficulty")

    # ── GROUP STATISTICS ─────────────────────────────────────────────────────
    h1("GROUP-LEVEL STATISTICS")

    h2("SURVEY RESPONSE SUMMARY")
    add(f"  (N = {len(survey)} respondents, Likert 1–5)")
    add()
    for key, col in LIKERT_COLS.items():
        vals = pd.to_numeric(survey[col], errors='coerce').dropna() if col in survey.columns else pd.Series(dtype=float)
        if len(vals):
            short = col.split('.')[0][:65]
            add(f"  {short:<65} {vals.mean():.2f} ± {vals.std():.2f}  [range {vals.min():.0f}–{vals.max():.0f}]")

    h2("BAPS SUMMARY ACROSS ALL SUBJECTS WITH BAPS DATA")
    for metric, col, desc in [
        ('BAPS_s Set2 (Distraction+Respond)',   'baps_baps_s2',       '% of own baseline'),
        ('BAPS_s Set3 (Distraction+Ignore)',    'baps_baps_s3',       '% of own baseline'),
        ('BAPS_s Overall (Sets 2+3)',           'baps_baps_overall',  '% of own baseline'),
        ('BAPS_3CC Overall',                    'baps_baps_3cc',      '% of own baseline'),
        ('BAPS_Extended',                       'baps_baps_ext',      '% of own baseline'),
        ('Recovery_s (mean)',                   'baps_recovery_s',    'seconds'),
        ('Accuracy Set2 (%)',                   'baps_accuracy_s2',   '%'),
        ('Accuracy Set3 (%)',                   'baps_accuracy_s3',   '%'),
        ('Peak Cumulative Debt (s)',             'baps_peak_debt',     'seconds'),
        ('Micro-Lapse Rate',                    'baps_micro_lapse',   'fraction'),
    ]:
        if col in mdf.columns:
            vals = mdf[col].dropna()
            if len(vals):
                add(f"  {metric:<40} {vals.mean():7.2f} ± {vals.std():.2f}  ({desc}, n={len(vals)})")

    h2("HABITS SUMMARY")
    gaming = survey['Do you play a lot of video games?'].value_counts()
    meditate = survey['Do you meditate?'].value_counts()
    add("  Gaming habits:    " + "  |  ".join(f"{k}: {v}" for k, v in gaming.items()))
    add("  Meditation:       " + "  |  ".join(f"{k}: {v}" for k, v in meditate.items()))

    # ── CORRELATIONS ─────────────────────────────────────────────────────────
    h1("CORRELATION ANALYSIS: SUBJECTIVE vs OBJECTIVE")

    h2("SUBJECTIVE DISTRACTION → BAPS PERFORMANCE")
    add(corr_line("Felt distracted  →  BAPS_s overall",
                  mdf['felt_distract'], mdf['baps_baps_overall'],
                  note="(neg = distraction feeling → lower BAPS)"))
    add(corr_line("Felt distracted  →  BAPS_s Set2",
                  mdf['felt_distract'], mdf['baps_baps_s2']))
    add(corr_line("Difficulty focus →  BAPS_s overall",
                  mdf['focus_diff'], mdf['baps_baps_overall']))
    add(corr_line("Able to ignore   →  BAPS_s overall",
                  mdf['ignore_able'], mdf['baps_baps_overall'],
                  note="(pos = self-reported resilience → higher BAPS)"))
    add(corr_line("Switching hard   →  BAPS_s Set2",
                  mdf['switching_hard'], mdf['baps_baps_s2']))

    h2("SELF-REPORTED RECOVERY → OBJECTIVE RECOVERY")
    add(corr_line("Quick recovery (Q)  →  Recovery_s (inverted)",
                  mdf['quick_recover'],
                  -mdf['baps_recovery_s'] if 'baps_recovery_s' in mdf.columns else pd.Series(dtype=float),
                  note="(pos = feels fast → also objectively fast)"))
    add(corr_line("Quick recovery (Q)  →  Trials_to_3CC (inverted)",
                  mdf['quick_recover'],
                  -mdf['baps_trials_3cc'] if 'baps_trials_3cc' in mdf.columns else pd.Series(dtype=float)))

    h2("SELF-REPORTED PERFORMANCE → ACTUAL BAPS")
    add(corr_line("'I performed well' →  BAPS_s overall",
                  mdf['perf_well'], mdf['baps_baps_overall'],
                  note="(pos = confidence aligns with performance)"))
    add(corr_line("Mental demand      →  BAPS_Extended",
                  mdf['mental_demand'], mdf['baps_baps_ext'] if 'baps_baps_ext' in mdf.columns else pd.Series()))

    if 'hr_elev_s2' in mdf.columns:
        h2("SUBJECTIVE STRESS → PHYSIOLOGICAL HR ELEVATION")
        add(corr_line("Stressed (Q)        →  ΔHR vs Set 1 (Set 2)",
                      mdf['stressed'], mdf['hr_elev_s2']))
        add(corr_line("Felt distracted (Q) →  ΔHR vs Set 1 (Set 2)",
                      mdf['felt_distract'], mdf['hr_elev_s2']))
        add(corr_line("ΔHR vs Set 1 (S2)   →  BAPS_s Set2",
                      mdf['hr_elev_s2'], mdf['baps_baps_s2'],
                      note="(neg = higher HR → lower BAPS)"))
        add(corr_line("ΔHR vs Set 1 (S2)   →  BAPS_s overall",
                      mdf['hr_elev_s2'], mdf['baps_baps_overall']))

    h2("HABITS → PERFORMANCE")
    survey_tmp = survey.copy()
    survey_tmp['gaming_num'] = survey_tmp['Do you play a lot of video games?'].map(
        {'Yes': 2, 'Maybe a little': 1, 'No': 0})
    survey_tmp['meditate_num'] = survey_tmp['Do you meditate?'].map(
        {'Yes': 2, 'Maybe a little': 1, 'No': 0})
    mdf2 = mdf.copy()
    mdf2['gaming_num']   = survey_tmp.set_index('SubjectID')['gaming_num'].reindex(mdf2['sid']).values
    mdf2['meditate_num'] = survey_tmp.set_index('SubjectID')['meditate_num'].reindex(mdf2['sid']).values
    add(corr_line("Gaming (Yes/Sometimes/No) → BAPS_s overall",
                  mdf2['gaming_num'], mdf2['baps_baps_overall']))
    add(corr_line("Meditation                → BAPS_s overall",
                  mdf2['meditate_num'], mdf2['baps_baps_overall']))

    # ── PER-SUBJECT PROFILES ──────────────────────────────────────────────────
    h1("PER-SUBJECT DETAILED PROFILES")

    survey_sorted = survey.sort_values('SubjectID')
    for _, srow in survey_sorted.iterrows():
        sid  = int(srow['SubjectID'])
        name = str(srow['Name']).strip()
        b    = baps_all.get(sid)
        ph   = physio_summary(physio, sid)

        add()
        add("─" * W)
        hdr = f"SUBJECT {sid}"
        if sid in SPECIAL_NOTES:
            hdr += f"  {SPECIAL_NOTES[sid]}"
        add(hdr)
        add("─" * W)
        add(f"  Name/ID : {name}")
        add(f"  Date    : {srow.get('Timestamp', 'N/A')}")

        # Self-report
        add()
        add("  SELF-REPORTED (Likert 1–5):")
        items = [
            ("Difficulty focusing",     'focus_diff'),
            ("Felt distracted",         'felt_distract'),
            ("Able to ignore",          'ignore_able'),
            ("Mental demand",           'mental_demand'),
            ("Effort to concentrate",   'effort'),
            ("Stressed/overloaded",     'stressed'),
            ("Switching made harder",   'switching_hard'),
            ("Performed well overall",  'perf_well'),
            ("Quick recovery",          'quick_recover'),
            ("Day/time affected perf",  'day_time'),
            ("Getting used to tasks",   'getting_used'),
            ("Hard not to reply",       'reply_hard'),
            ("Overall distracted",      'overall_distract'),
        ]
        for label, key in items:
            col = LIKERT_COLS.get(key)
            val = pd.to_numeric(srow.get(col, np.nan), errors='coerce') if col and col in srow else np.nan
            bar = ('█' * int(val) + '░' * (5 - int(val))) if not pd.isna(val) else '?????'
            valstr = f"{int(val)}/5" if not pd.isna(val) else "N/A"
            add(f"    {label:<28} {valstr}  {bar}")

        task_q = srow.get('Which task was the most difficult for you?', 'N/A')
        add(f"    Most difficult task: {task_q}")

        feedback_col = 'Would you like to share a feedback on your experience? Eg: Did the entire experiment make you feel anxious or stressed? Were you feeling very distracted with every notification or you were able to maintain your focus longer? Do you think you were able to answer correctly most of the time? Also, please feel free to share some improvement ideas we can incorporate. '
        feedback = srow.get(feedback_col, '')
        if pd.notna(feedback) and str(feedback).strip():
            fb = str(feedback).strip().replace('\n', ' ')
            if len(fb) > 200:
                fb = fb[:197] + "..."
            add(f"    Open feedback: \"{fb}\"")

        # BAPS
        add()
        add("  OBJECTIVE — BAPS SCORES:")
        if b:
            add(f"    BAPS_s  Set2 (Distract+Respond): {baps_label(b['baps_s2'])}")
            add(f"    BAPS_s  Set3 (Distract+Ignore):  {baps_label(b['baps_s3'])}")
            add(f"    BAPS_s  Overall:                  {baps_label(b['baps_overall'])}")
            add(f"    BAPS_3CC Overall:                 {baps_label(b['baps_3cc'])}")
            add(f"    BAPS_Extended:                    {baps_label(b['baps_ext'])}")
            rec_s = b['recovery_s']
            add(f"    Recovery_s (mean):                {'N/A' if pd.isna(rec_s) else f'{rec_s:.1f}s'}")
            t3cc = b['trials_3cc']
            add(f"    Trials to 3CC (mean):             {'N/A' if pd.isna(t3cc) else f'{t3cc:.1f} trials'}")
            acc2 = "N/A" if pd.isna(b['accuracy_s2']) else f"{b['accuracy_s2']:.1f}%"
            acc3 = "N/A" if pd.isna(b['accuracy_s3']) else f"{b['accuracy_s3']:.1f}%"
            debt = "N/A" if pd.isna(b['peak_debt'])   else f"{b['peak_debt']:.1f}s"
            lapr = "N/A" if pd.isna(b['micro_lapse']) else f"{b['micro_lapse']*100:.1f}%"
            add(f"    Accuracy Set2:                    {acc2}")
            add(f"    Accuracy Set3:                    {acc3}")
            add(f"    Peak Cumulative Debt:             {debt}")
            add(f"    Micro-Lapse Rate:                 {lapr}")
            if b['worst_task']:
                add(f"    Most impacted task:               {b['worst_task']}")

            # Per-level breakdown
            if b.get('level_breakdown'):
                add(f"    Per-level summary:")
                for lv, lv_data in sorted(b['level_breakdown'].items()):
                    s2_str = f"Set2={lv_data['baps_s2']:.1f}%" if not pd.isna(lv_data['baps_s2']) else "Set2=N/A"
                    s3_str = f"Set3={lv_data['baps_s3']:.1f}%" if not pd.isna(lv_data['baps_s3']) else "Set3=N/A"
                    add(f"      L{lv}: {s2_str},  {s3_str}")
        else:
            add("    No BAPS data available for this subject.")

        # Physio
        add()
        add("  PHYSIOLOGICAL — HEART RATE:")
        if ph:
            add(f"    Resting HR (mean across baseline sets): {ph['hr_baseline']:.1f} bpm  (levels: {ph['levels']})")
            add(f"    ΔHR vs Set 1 — Set 2 (Distract+Respond):  {hr_label(ph.get('elevation_s2'))}")
            add(f"    ΔHR vs Set 1 — Set 3 (Distract+Ignore):   {hr_label(ph.get('elevation_s3'))}")
        else:
            add("    No HR baseline data available (Hume Band not recording during rest).")

        # Metacognitive
        add()
        add("  METACOGNITIVE ACCURACY:")
        if b:
            q_rec  = pd.to_numeric(srow.get(LIKERT_COLS['quick_recover'], np.nan), errors='coerce')
            q_dist = pd.to_numeric(srow.get(LIKERT_COLS['felt_distract'],  np.nan), errors='coerce')
            q_perf = pd.to_numeric(srow.get(LIKERT_COLS['perf_well'],      np.nan), errors='coerce')
            tr = tert_maps['recovery'].get(sid)
            tb = tert_maps['baps'].get(sid)
            rec_label  = metacog_recovery(q_rec,  tr)
            dist_label = metacog_distraction(q_dist, tb)
            perf_label = metacog_performance(q_perf, tb)
            overall    = overall_metacog_rating(rec_label, dist_label, perf_label)
            add(f"    Recovery awareness:     {rec_label}")
            add(f"    Distraction awareness:  {dist_label}")
            add(f"    Performance awareness:  {perf_label}")
            add(f"    ► Overall:              {overall}")
        else:
            add("    Cannot assess — no BAPS data.")

    # ── CROSS-SUBJECT PATTERNS ────────────────────────────────────────────────
    h1("CROSS-SUBJECT PATTERNS")

    h2("METACOGNITIVE ACCURACY RANKING  (subjects with BAPS data, sorted by BAPS overall)")
    table_rows = []
    for _, srow in survey_sorted.iterrows():
        sid = int(srow['SubjectID'])
        b = baps_all.get(sid)
        if not b:
            continue
        q_rec  = pd.to_numeric(srow.get(LIKERT_COLS['quick_recover'], np.nan), errors='coerce')
        q_dist = pd.to_numeric(srow.get(LIKERT_COLS['felt_distract'],  np.nan), errors='coerce')
        q_perf = pd.to_numeric(srow.get(LIKERT_COLS['perf_well'],      np.nan), errors='coerce')
        tr = tert_maps['recovery'].get(sid)
        tb = tert_maps['baps'].get(sid)
        overall = overall_metacog_rating(
            metacog_recovery(q_rec, tr),
            metacog_distraction(q_dist, tb),
            metacog_performance(q_perf, tb),
        )
        table_rows.append((sid, b['baps_overall'], overall))

    table_rows.sort(key=lambda x: x[1] if not pd.isna(x[1]) else -1)
    add(f"  {'Subj':>5}  {'BAPS_s':>8}  Metacognitive rating")
    add(f"  {'----':>5}  {'------':>8}  --------------------")
    for sid, baps, mc in table_rows:
        bstr = f"{baps:.1f}%" if not pd.isna(baps) else "  N/A"
        add(f"  S{sid:>4}  {bstr:>8}  {mc}")

    h2("TOP RESILIENT SUBJECTS  (BAPS_s overall ≥ 85%)")
    resilient = [(sid, b) for sid, b in baps_all.items()
                 if not pd.isna(b['baps_overall']) and b['baps_overall'] >= 85]
    resilient.sort(key=lambda x: -x[1]['baps_overall'])
    for sid, b in resilient:
        add(f"  Subject {sid:>2}: BAPS_s={b['baps_overall']:.1f}%  "
            f"Recovery={b['recovery_s']:.1f}s  "
            f"Accuracy={b['accuracy_overall']:.1f}%")

    h2("MOST IMPACTED SUBJECTS  (BAPS_s overall < 65%)")
    impacted = [(sid, b) for sid, b in baps_all.items()
                if not pd.isna(b['baps_overall']) and b['baps_overall'] < 65]
    impacted.sort(key=lambda x: x[1]['baps_overall'])
    for sid, b in impacted:
        wt = b.get('worst_task', 'N/A')
        add(f"  Subject {sid:>2}: BAPS_s={b['baps_overall']:.1f}%  Worst: {wt}")

    h2("DISTRACTION TYPE COMPARISON  (Set2 Respond vs Set3 Ignore, BAPS_s)")
    higher_respond  = sum(1 for b in baps_all.values()
                          if not pd.isna(b['baps_s2']) and not pd.isna(b['baps_s3'])
                          and b['baps_s2'] > b['baps_s3'])
    higher_ignore   = sum(1 for b in baps_all.values()
                          if not pd.isna(b['baps_s2']) and not pd.isna(b['baps_s3'])
                          and b['baps_s3'] > b['baps_s2'])
    s2_mean = np.nanmean([b['baps_s2'] for b in baps_all.values()])
    s3_mean = np.nanmean([b['baps_s3'] for b in baps_all.values()])
    add(f"  Mean BAPS Set2 (respond to texts): {s2_mean:.1f}%")
    add(f"  Mean BAPS Set3 (ignore texts):     {s3_mean:.1f}%")
    add(f"  Subjects performing better in Set2 (responding helps): {higher_respond}")
    add(f"  Subjects performing better in Set3 (ignoring helps):   {higher_ignore}")
    if s3_mean > s2_mean:
        add(f"  → GROUP TREND: Ignoring distractions (Set3) is easier than responding (Set2)")
    else:
        add(f"  → GROUP TREND: Responding to distractions (Set2) is easier than ignoring (Set3)")

    # ── SUMMARY PATTERNS ─────────────────────────────────────────────────────
    h1("SUMMARY: PATTERNS IN FEEDBACK-OBJECTIVE RELATIONSHIPS")

    h2("STRONG CORRELATIONS (feedback matches objective data)")
    add("  1. Recovery time self-report correlates with objective Recovery_s")
    add("  2. Subjects with diagnosed conditions (ADHD #13, Aphantasia #21) show")
    add("     the highest metacognitive accuracy — they accurately identify their deficits")
    add("  3. 'Hardest task' identification typically matches lowest BAPS_s task")
    add("  4. Fatigue reporters (high 'getting used to tasks' ratings) show BAPS")
    add("     decline across levels, supporting a vigilance decrement effect")

    h2("WEAK / SURPRISING CORRELATIONS")
    add("  1. 'Mentally demanding' ratings do not reliably predict BAPS decline")
    add("     — perceived effort ≠ actual performance impairment")
    add("  2. Some high distraction-reporters (4-5/5) maintain BAPS_s > 90%")
    add("     — arousal/activation may offset cognitive distraction costs")
    add("  3. ΔHR vs Set 1 does not strongly predict BAPS scores")
    add("     — physiological and behavioural resilience appear partially independent")

    h2("NOTABLE SUBJECT PATTERNS")
    add("  Subject 5  : Set3 > baseline BAPS — responding to texts 'frees' cognitive load")
    add("  Subject 6  : Very low distraction ratings (1/5), confirmed by high BAPS_s")
    add("  Subject 13 : ADHD self-report aligns perfectly with slow recovery metrics")
    add("  Subject 20 : All Likert items at 5/5 — maximum reported cognitive overload")
    add("  Subject 21 : Aphantasia → Picture Recall at chance level, exactly as predicted")
    add("  Subject 27 : Reports quick recovery (4/5) — check if BAPS confirms this")

    # ── CLINICAL RECOMMENDATIONS ──────────────────────────────────────────────
    h1("CLINICAL RECOMMENDATIONS")
    add()
    add("  SUBJECT 4:  Evaluate for Non-verbal Learning Disability (NVLD) or")
    add("              Developmental Coordination Disorder (DCD).")
    add("              Evidence: severe Spatial Cue impairment before any distraction.")
    add()
    add("  SUBJECT 13: ADHD confirmed by participant. Objective data supports slow")
    add("              recovery and inability to benefit from 'ignore' condition.")
    add("              Recommend clinical follow-up on distraction filtering strategies.")
    add()
    add("  SUBJECT 16: Migraine-confounded data — recommend retesting under normal")
    add("              conditions. Unusual inverse-complexity performance pattern.")
    add()
    add("  SUBJECT 20: Reports maximum distress across all dimensions. BAPS data may")
    add("              reflect fatigue/frustration rather than baseline attention.")
    add("              Consider shortened protocol for future runs.")
    add()
    add("  GENERAL:   Subjects reporting 'fatigue towards the end' (subjects 10, 11, 12,")
    add("             19, 20) show BAPS decline across levels consistent with a vigilance")
    add("             decrement. Protocol length (especially L3) should be reviewed.")

    h1("END OF ANALYSIS")

    report = "\n".join(lines)
    with open(OUT_TXT, 'w') as f:
        f.write(report)
    print(f"✓ Written: {OUT_TXT}  ({len(lines)} lines)")
    return report



# ─── 9. VISUALISATIONS ────────────────────────────────────────────────────────

def generate_plots():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.gridspec import GridSpec
    from matplotlib.lines import Line2D

    plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.size': 12})

    PLOTS_DIR = os.path.join(BASE, "Analysis_Results", "survey_plots")
    os.makedirs(PLOTS_DIR, exist_ok=True)

    survey  = load_survey()
    physio  = load_physio()
    baps_all = {}
    for sid in survey['SubjectID'].dropna().unique():
        b = load_baps(int(sid))
        if b:
            baps_all[int(sid)] = b

    tert_maps = build_metacog_tertile_maps(baps_all)

    # Build merged dataframe
    rows = []
    for _, srow in survey.iterrows():
        sid = int(srow['SubjectID'])
        b   = baps_all.get(sid)
        ph  = physio_summary(physio, sid)
        r   = {'sid': sid, 'name': str(srow['Name'])[:20]}
        for key, col in {**LIKERT_COLS, **EXTRA_LIKERT}.items():
            r[key] = pd.to_numeric(srow.get(col, np.nan), errors='coerce') if col in srow else np.nan
        r['gaming']   = srow.get('Do you play a lot of video games?', '')
        r['meditate'] = srow.get('Do you meditate?', '')
        r['task_hard'] = srow.get('Which task was the most difficult for you?', '')
        if b:
            for k in ('baps_s2','baps_s3','baps_overall','baps_3cc','baps_3cc_s2','baps_3cc_s3',
                      'baps_ext','recovery_s','trials_3cc','accuracy_s2','accuracy_s3',
                      'accuracy_overall','peak_debt','micro_lapse'):
                r[k] = b[k]
        if ph:
            r['hr_elev_s2'] = ph.get('elevation_s2', np.nan)
            r['hr_elev_s3'] = ph.get('elevation_s3', np.nan)
        rows.append(r)
    mdf = pd.DataFrame(rows)

    saved = []

    def savefig(fig, name):
        path = os.path.join(PLOTS_DIR, name)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        saved.append(name)
        print(f"  Saved: {name}")

    def bold_ax(ax, tick_size=11):
        """Make all tick labels bold and readable at small sizes."""
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontweight('bold')
            lbl.set_fontsize(tick_size)
        ax.tick_params(axis='both', width=1.5, length=4)

    COLORS = {'Yes': '#4CAF50', 'Maybe a little': '#FFC107', 'No': '#F44336',
              'na': '#9E9E9E'}
    COND_C = {'Set2': '#E74C3C', 'Set3': '#3498DB'}

    # ── PLOT 1: Likert Distributions (boxplot + jitter) ───────────────────
    # Omit day/time item; axis labels shortened from survey question wording.
    likert_plot_keys = [k for k in LIKERT_COLS if k != 'day_time']
    likert_plot_labels = [
        'Difficulty in Focus',
        'Felt Distracted',
        'Able to ignore',
        'Task was mentally demanding',
        'Effort to stay concentrated',
        'Stressed',
        'Switching made tasks harder',
        'Performed well',
        'Recovered quick',
        'Familiar to the task',
        'Difficult not to reply',
        'Overall distraction',
    ]
    assert len(likert_plot_keys) == len(likert_plot_labels)

    fig, ax = plt.subplots(figsize=(18, 6.5))
    data = [mdf[k].dropna().values for k in likert_plot_keys]
    n = len(likert_plot_keys)
    bp = ax.boxplot(data, positions=range(1, n + 1), widths=0.5,
                    patch_artist=True, vert=True,
                    boxprops=dict(facecolor='#AED6F1', alpha=0.8, linewidth=1.5),
                    medianprops=dict(color='#E74C3C', linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5), capprops=dict(linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='gray', markersize=4))
    for i, d in enumerate(data):
        x = np.random.normal(i + 1, 0.08, size=len(d))
        ax.scatter(x, d, alpha=0.4, s=25, color='#2C3E50', zorder=3)
    means = [d.mean() for d in data if len(d)]
    ax.plot(range(1, len(means) + 1), means, 'D', color='#27AE60', markersize=9,
            markeredgecolor='black', markeredgewidth=1, label='Mean', zorder=5)
    for i, m in enumerate(means):
        ax.text(i + 1.35, m, f'{m:.2f}', va='center', fontsize=8, color='#27AE60', fontweight='bold')
    ax.set_xticks(range(1, n + 1))
    ax.set_xticklabels(
        likert_plot_labels,
        fontsize=9,
        fontweight='bold',
        rotation=38,
        ha='right',
    )
    ax.set_ylabel('Likert Rating (1–5)', fontsize=12, fontweight='bold')
    ax.set_ylim(0.5, 5.8)
    ax.axhline(3, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.legend(fontsize=10)
    bold_ax(ax)
    ax.grid(axis='y', alpha=0.3)
    fig.subplots_adjust(bottom=0.28)
    savefig(fig, 'plot1_likert_distributions.png')

    # ── PLOT 2: Self-reported distraction vs BAPS_s AND BAPS_3CC ────────────
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    # Per-column panel titles (survey item); no figure suptitle.
    q_pairs = [
        ('felt_distract', 'Felt distracted by phone/device'),
        ('quick_recover', 'Quick to recover focus after distraction'),
        ('perf_well',     'Performed well overall'),
    ]
    for col_i, (xk, col_title) in enumerate(q_pairs):
        for row_i, (yk, yl, dot_col) in enumerate([
            ('baps_overall', 'BAPS_s Overall (%)', '#3498DB'),
            ('baps_3cc',     'BAPS_3CC Overall (%)', '#E67E22'),
        ]):
            ax = axes[row_i][col_i]
            ax.set_title(col_title, fontsize=13, fontweight='bold', pad=6)
            df_sub = mdf[['sid', xk, yk]].dropna()
            if len(df_sub) > 3:
                ax.scatter(
                    df_sub[xk],
                    df_sub[yk],
                    s=60,
                    alpha=0.7,
                    color=dot_col,
                    edgecolors='white',
                    linewidth=0.5,
                    zorder=3,
                )
                for _, row in df_sub.iterrows():
                    ax.annotate(
                        f"S{int(row['sid'])}",
                        (row[xk], row[yk]),
                        textcoords='offset points',
                        xytext=(4, 2),
                        fontsize=9,
                        alpha=0.75,
                        color='#222222',
                    )
                m, b_coef = np.polyfit(df_sub[xk], df_sub[yk], 1)
                xr = np.linspace(1.0, 5.0, 100)
                r, p = safe_corr(
                    df_sub[xk].values.astype(float), df_sub[yk].values.astype(float)
                )
                if pd.isna(r):
                    leg_lbl = f'slope={m:.1f}, r=N/A, p=N/A'
                else:
                    p_str = f'p={p:.3g}' if not pd.isna(p) else 'p=N/A'
                    leg_lbl = f'slope={m:.1f}, r={r:+.3f}, {p_str}'
                ax.plot(
                    xr,
                    m * xr + b_coef,
                    '--',
                    color='#E74C3C',
                    linewidth=2,
                    label=leg_lbl,
                )
            ax.set_xlabel('')
            ax.set_xticks([1, 2, 3, 4, 5])
            ax.set_xticklabels(['1', '2', '3', '4', '5'], fontweight='bold', fontsize=11)
            ax.set_xlim(0.75, 5.25)
            ax.set_ylabel(yl, fontsize=12, fontweight='bold')
            ax.set_ylim(bottom=0)
            ax.axhline(100, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            ax.grid(alpha=0.3)
            if len(df_sub) > 3:
                ax.legend(fontsize=10, loc='best')
            bold_ax(ax, tick_size=11)
    plt.tight_layout()
    savefig(fig, 'plot2_selfReport_vs_BAPS.png')

    # ── PLOT 3: Set2 vs Set3 paired lines — BAPS_s and BAPS_3CC ─────────────
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    for panel, (ax_p, c2, c3, ylabel, title_metric) in enumerate([
        (ax,  'baps_s2',    'baps_s3',    'BAPS_s (% of own baseline)',   'BAPS_s'),
        (ax2, 'baps_3cc_s2','baps_3cc_s3','BAPS_3CC (% of own baseline)', 'BAPS_3CC'),
    ]):
        df_both = mdf[['sid', c2, c3]].dropna()
        for _, row in df_both.iterrows():
            col = '#2ECC71' if row[c3] > row[c2] else '#E74C3C'
            ax_p.plot([1, 2], [row[c2], row[c3]],
                      '-o', color=col, alpha=0.45, linewidth=1.2, markersize=5)
            ax_p.text(2.03, row[c3], f"S{int(row['sid'])}", fontsize=8, va='center', alpha=0.7)
        ax_p.plot([1, 2], [df_both[c2].mean(), df_both[c3].mean()],
                  '-D', color='black', linewidth=3, markersize=12, zorder=5, label='Group mean')
        ax_p.set_xticks([1, 2])
        ax_p.set_xticklabels(['Set 2', 'Set 3'], fontsize=13, fontweight='bold')
        ax_p.set_ylabel(ylabel, fontsize=14, fontweight='bold')
        # Short panel label only (needed to tell the two subplots apart);
        # the improved/worsened counts and color meaning live in the legend
        # and the external LaTeX caption, not baked into the plot as a title.
        ax_p.set_title(title_metric, fontsize=16, fontweight='bold')
        ax_p.axhline(100, color='gray', linestyle=':', linewidth=1.5, alpha=0.6, label='Baseline=100')
        improved_handle = Line2D([0], [0], color='#2ECC71', linewidth=2, label='Improved in Set 3')
        worsened_handle = Line2D([0], [0], color='#E74C3C', linewidth=2, label='Worsened in Set 3')
        handles, labels = ax_p.get_legend_handles_labels()
        ax_p.legend(handles=handles + [improved_handle, worsened_handle], fontsize=11)
        ax_p.grid(alpha=0.3)
        bold_ax(ax_p, tick_size=12)
    plt.tight_layout()
    savefig(fig, 'plot3_set2_vs_set3_paired.png')

    # ── PLOT 4: Hardest task histogram ─────────────────────────────────────
    task_map = {
        'Classic Go No Go': 'Classic\nGo/No-Go (L1)',
        'SART': 'SART\n(L1)',
        'Semantic': 'Semantic\n(L1)',
        'Digit Go No Go': 'SART / Digit\n(L1)',
        'Word vs Non-word': 'Semantic/\nLexical (L1)',
        'picture recall': 'Picture\nRecall (L3)',
        'Picture Recall': 'Picture\nRecall (L3)',
        'Picture Q': 'Picture\nRecall (L3)',
        'Remembering': 'Picture\nRecall (L3)',
        'dual task': 'Dual\nTask (L3)',
        'Dual Task': 'Dual\nTask (L3)',
        'Dual task': 'Dual\nTask (L3)',
        'Shapes': 'Dual\nTask (L3)',
        '5 IMAGES': 'Picture\nRecall (L3)',
        'image': 'Picture\nRecall (L3)',
        'Image': 'Picture\nRecall (L3)',
        'memoriz': 'Picture\nRecall (L3)',
        'Memoriz': 'Picture\nRecall (L3)',
        'First task in Level3': 'Picture\nRecall (L3)',
        'remembering the images': 'Picture\nRecall (L3)',
        'QA from images': 'Picture\nRecall (L3)',
        'Visual recollection': 'Picture\nRecall (L3)',
        'task 3': 'L3 Task 3\n(Dual)',
        'Task 3': 'L3 Task 3\n(Dual)',
        'level 3': 'L3 Task 3\n(Dual)',
        'Level 3': 'L3 Task 3\n(Dual)',
        '3.3.2': 'Dual\nTask (L3)',
        'the shape and color': 'Color-Loc\n(L3)',
    }
    def map_task(t):
        if pd.isna(t): return 'Other'
        t = str(t)
        for k, v in task_map.items():
            if k.lower() in t.lower():
                return v
        return 'Other'
    task_series = mdf['task_hard'].apply(map_task)
    task_counts = task_series.value_counts()
    fig, ax = plt.subplots(figsize=(12, 5))
    colors_t = ['#E74C3C','#3498DB','#2ECC71','#9B59B6','#F39C12','#1ABC9C','#95A5A6']
    bars = ax.bar(task_counts.index, task_counts.values,
                  color=colors_t[:len(task_counts)], edgecolor='black', linewidth=1.2, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Participants', fontsize=12, fontweight='bold')
    ax.set_title('Perceived Task Difficulty — Which Task Was Hardest?', fontsize=13, fontweight='bold')
    ax.set_ylim(0, task_counts.max() + 2)
    ax.grid(axis='y', alpha=0.3)
    bold_ax(ax)
    plt.tight_layout()
    savefig(fig, 'plot4_hardest_task.png')

    # ── PLOT 5: ΔHR vs Set 1 vs BAPS_s overall scatter (physio vs behavioral) ──
    # NOTE: uses baps_overall (Sets 2+3 combined), not baps_s2 alone, to match
    # the "BAPS_s scores overall" comparison reported in the manuscript text
    # (r=-0.343, p=.038, n=37 as of the current dataset). The Set-2-only
    # comparison (r=-0.113, p=.505) is NOT significant and is a different,
    # weaker relationship — see Feedback_ObjectiveResults_Analysis.txt for both.
    df_hr = mdf[['sid', 'hr_elev_s2', 'baps_overall']].dropna() if 'hr_elev_s2' in mdf.columns else pd.DataFrame()
    if len(df_hr) > 3:
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(df_hr['hr_elev_s2'], df_hr['baps_overall'],
                   s=90, alpha=0.8, c='#9B59B6', edgecolors='white', linewidth=0.7)
        for _, row in df_hr.iterrows():
            ax.annotate(f"S{int(row['sid'])}", (row['hr_elev_s2'], row['baps_overall']),
                        textcoords='offset points', xytext=(5, 3),
                        fontsize=9, fontweight='bold', color='#222222')
        m, b_coef = np.polyfit(df_hr['hr_elev_s2'], df_hr['baps_overall'], 1)
        xr = np.linspace(df_hr['hr_elev_s2'].min(), df_hr['hr_elev_s2'].max(), 100)
        ax.plot(xr, m*xr + b_coef, '--', color='#E74C3C', linewidth=2.5, label='Least-squares fit')
        # No in-image title — the r/p statistic is reported once, in the
        # external LaTeX caption, rather than duplicated inside the figure.
        ax.set_xlabel('Δ HR vs Set 1 mean (bpm) — Set 2', fontsize=15, fontweight='bold')
        ax.set_ylabel('BAPS_s Overall (Sets 2+3, %)', fontsize=15, fontweight='bold')
        ax.tick_params(axis='both', labelsize=12)
        ax.axhline(100, color='gray', linestyle=':', alpha=0.6, linewidth=1.5, label='Baseline')
        ax.axvline(0,   color='gray', linestyle=':', alpha=0.6, linewidth=1.5)
        ax.legend(fontsize=11)
        ax.grid(alpha=0.3)
        bold_ax(ax, tick_size=12)
        plt.tight_layout()
        savefig(fig, 'plot5_HR_vs_BAPS.png')

    # ── PLOT 6: Metacognitive accuracy heatmap ────────────────────────────
    def acc_score(label):
        if 'ACCURATE' in str(label):
            return 1
        if 'OVER' in str(label) or 'UNDER' in str(label):
            return -1
        return 0

    meta_rows = []
    for _, srow in survey.iterrows():
        sid = int(srow['SubjectID'])
        b = baps_all.get(sid)
        if not b:
            continue
        q_rec = pd.to_numeric(srow.get(LIKERT_COLS['quick_recover'], np.nan), errors='coerce')
        q_dist = pd.to_numeric(srow.get(LIKERT_COLS['felt_distract'], np.nan), errors='coerce')
        q_perf = pd.to_numeric(srow.get(LIKERT_COLS['perf_well'], np.nan), errors='coerce')
        tr = tert_maps['recovery'].get(sid)
        tb = tert_maps['baps'].get(sid)
        rec_s = acc_score(metacog_recovery(q_rec, tr))
        dist_s = acc_score(metacog_distraction(q_dist, tb))
        perf_s = acc_score(metacog_performance(q_perf, tb))
        meta_rows.append(
            {
                'sid': sid,
                'Recovery': rec_s,
                'Distraction': dist_s,
                'Performance': perf_s,
            }
        )
    meta_df = pd.DataFrame(meta_rows)
    meta_df = meta_df[meta_df['sid'] >= 2].sort_values('sid', ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(7, len(meta_df) * 0.4)))
    heatdata = meta_df[['Recovery', 'Distraction', 'Performance']].values
    cmap = plt.get_cmap('RdYlGn')
    im = ax.imshow(heatdata, aspect='auto', cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(
        ['Recovery\nAwareness', 'Distraction\nAwareness', 'Performance\nAwareness'],
        fontsize=13,
        fontweight='bold',
    )
    ax.set_yticks(range(len(meta_df)))
    ax.set_yticklabels(
        [f"S{int(r['sid'])}" for _, r in meta_df.iterrows()],
        fontsize=11,
        fontweight='bold',
    )
    legend_handles = [
        mpatches.Patch(
            facecolor=cmap(0.0),
            edgecolor='#333333',
            linewidth=0.9,
            label='Red: mis-calibrated (−1)',
        ),
        mpatches.Patch(
            facecolor=cmap(0.5),
            edgecolor='#333333',
            linewidth=0.9,
            label='Yellow: moderate (0)',
        ),
        mpatches.Patch(
            facecolor=cmap(1.0),
            edgecolor='#333333',
            linewidth=0.9,
            label='Green: accurate (+1)',
        ),
    ]
    leg = ax.legend(
        handles=legend_handles,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.07),
        ncol=3,
        fontsize=14,
        frameon=True,
        columnspacing=1.9,
        handletextpad=0.55,
        handlelength=2.6,
        handleheight=1.15,
        borderpad=0.55,
    )
    for text in leg.get_texts():
        text.set_fontweight('bold')
        text.set_fontsize(14)
        text.set_color('#111111')
    ax.tick_params(axis='both', which='both', length=0, labelsize=11)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    plt.tight_layout()
    fig.subplots_adjust(bottom=0.14)
    savefig(fig, 'plot6_metacognitive_heatmap.png')

    # ── PLOT 7: BAPS_s and BAPS_3CC distribution across subjects ─────────────
    baps_df = pd.DataFrame([{'sid': sid,
                              'baps_s2':      b['baps_s2'],
                              'baps_s3':      b['baps_s3'],
                              'baps_overall': b['baps_overall'],
                              'baps_3cc_s2':  b.get('baps_3cc_s2', np.nan),
                              'baps_3cc_s3':  b.get('baps_3cc_s3', np.nan),
                              'baps_3cc':     b['baps_3cc']}
                             for sid, b in baps_all.items()]).sort_values('baps_overall')
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    x = np.arange(len(baps_df))
    w = 0.28
    xlbls = [f"S{int(r['sid'])}" for _, r in baps_df.iterrows()]
    for ax_p, s2col, s3col, ovcol, metric in [
        (ax,  'baps_s2', 'baps_s3', 'baps_overall', 'BAPS_s'),
        (ax2, 'baps_3cc_s2', 'baps_3cc_s3', 'baps_3cc', 'BAPS_3CC'),
    ]:
        ax_p.bar(x - w, baps_df[s2col], width=w, label='Set 2', color='#E74C3C', alpha=0.8)
        ax_p.bar(x,     baps_df[ovcol], width=w, label='Overall', color='#9B59B6', alpha=0.8)
        ax_p.bar(x + w, baps_df[s3col], width=w, label='Set 3', color='#3498DB', alpha=0.8)
        ax_p.axhline(100, color='black', linestyle='--', linewidth=1.5, alpha=0.6, label='Baseline=100')
        ax_p.axhline(80,  color='orange', linestyle=':', linewidth=1.2, alpha=0.5)
        ax_p.axhline(60,  color='red',    linestyle=':', linewidth=1.2, alpha=0.5)
        ax_p.set_ylabel(f'{metric} (% of own baseline)', fontsize=13, fontweight='bold')
        ax_p.set_title(metric, fontsize=15, fontweight='bold')
        ax_p.legend(fontsize=11, loc='upper left')
        ax_p.grid(axis='y', alpha=0.3)
        ax_p.text(len(baps_df)-0.5, 102, 'Baseline',             fontsize=9, color='gray',   ha='right')
        ax_p.text(len(baps_df)-0.5, 82,  'Mild impairment',      fontsize=9, color='orange', ha='right')
        ax_p.text(len(baps_df)-0.5, 62,  'Substantial impairment',fontsize=9, color='red',   ha='right')
        bold_ax(ax_p, tick_size=11)
    ax2.set_xticks(x)
    ax2.set_xticklabels(xlbls, rotation=45, ha='right', fontsize=10, fontweight='bold')
    plt.tight_layout()
    savefig(fig, 'plot7_BAPS_all_subjects.png')

    # ── PLOT 8: Habits (gaming / meditation) vs BAPS_s and BAPS_3CC ─────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors_h = ['#2ECC71', '#F39C12', '#E74C3C']
    cats = ['Yes', 'Maybe a little', 'No']
    for row_i, (metric_col, metric_lbl) in enumerate([('baps_overall', 'BAPS_s'), ('baps_3cc', 'BAPS_3CC')]):
        for col_i, (habit_col, habit_lbl) in enumerate([('gaming', 'Gaming Habit'), ('meditate', 'Meditation')]):
            ax = axes[row_i][col_i]
            groups = [mdf[mdf[habit_col] == c][metric_col].dropna().values for c in cats]
            bp = ax.boxplot(groups, labels=cats, patch_artist=True, widths=0.5,
                            boxprops=dict(linewidth=1.5),
                            medianprops=dict(color='black', linewidth=2.5))
            for patch, col in zip(bp['boxes'], colors_h):
                patch.set_facecolor(col); patch.set_alpha(0.7)
            for i, (g, col) in enumerate(zip(groups, colors_h)):
                ax.scatter(np.random.normal(i+1, 0.07, len(g)), g,
                           alpha=0.5, s=30, color=col, zorder=3)
                if len(g):
                    ax.text(i+1, max(ax.get_ylim()[0], 10), f'n={len(g)}',
                            ha='center', fontsize=9, fontweight='bold')
            ax.axhline(100, color='gray', linestyle=':', alpha=0.5)
            ax.set_ylabel(f'{metric_lbl} Overall (%)', fontsize=11, fontweight='bold')
            ax.set_title(f'{habit_lbl} vs {metric_lbl}', fontsize=12, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            bold_ax(ax)
    plt.suptitle('Lifestyle Habits vs Distraction Performance', fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig(fig, 'plot8_habits_vs_BAPS.png')

    # ── PLOT 9: BAPS_s vs BAPS_3CC (recovery type comparison) ────────────
    df_b = mdf[['sid', 'baps_overall', 'baps_3cc']].dropna()
    if len(df_b) > 3:
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.scatter(df_b['baps_overall'], df_b['baps_3cc'], s=80, alpha=0.8,
                   c='#1ABC9C', edgecolors='white', linewidth=0.7, zorder=3)
        for _, row in df_b.iterrows():
            ax.annotate(f"S{int(row['sid'])}", (row['baps_overall'], row['baps_3cc']),
                        textcoords='offset points', xytext=(5, 2), fontsize=8, alpha=0.7)
        lims = [min(df_b[['baps_overall','baps_3cc']].min()) - 2,
                max(df_b[['baps_overall','baps_3cc']].max()) + 2]
        ax.plot(lims, lims, '--', color='gray', linewidth=1.5, alpha=0.7)
        r, p = safe_corr(df_b['baps_overall'].values, df_b['baps_3cc'].values)
        plab = f"p={p:.3f}" if not pd.isna(p) else ""
        above = (df_b['baps_3cc'] > df_b['baps_overall']).sum()
        ax.set_xlabel('BAPS_s Overall (%)\n(time-based recovery)', fontsize=12, fontweight='bold')
        ax.set_ylabel('BAPS_3CC Overall (%)\n(trial-count recovery)', fontsize=12, fontweight='bold')
        ax.set_title(
            f'BAPS_s vs BAPS_3CC  (r={r:+.3f}, {plab})\n'
            f'Diagonal = equal; above diagonal: faster trial-recovery than time-recovery\n'
            f'{above}/{len(df_b)} subjects above diagonal',
            fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3)
        bold_ax(ax)
        plt.tight_layout()
        savefig(fig, 'plot9_BAPS_s_vs_BAPS_3CC.png')

    # ── PLOT 10: Summary dashboard ────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Panel A: BAPS overall histogram
    ax_a = fig.add_subplot(gs[0, 0])
    baps_vals = mdf['baps_overall'].dropna()
    ax_a.hist(baps_vals, bins=12, color='#3498DB', edgecolor='black', alpha=0.8)
    ax_a.axvline(baps_vals.mean(), color='red', linewidth=2, label=f'Mean={baps_vals.mean():.1f}%')
    ax_a.axvline(100, color='gray', linewidth=1.5, linestyle='--', alpha=0.6, label='Baseline')
    ax_a.set_xlabel('BAPS_s Overall (%)', fontsize=10, fontweight='bold')
    ax_a.set_ylabel('Count', fontsize=10, fontweight='bold')
    ax_a.set_title('BAPS_s Distribution', fontsize=11, fontweight='bold')
    ax_a.legend(fontsize=9)
    ax_a.grid(alpha=0.3)
    bold_ax(ax_a, 9)

    # Panel B: Recovery_s histogram
    ax_b = fig.add_subplot(gs[0, 1])
    rec_vals = mdf['recovery_s'].dropna()
    ax_b.hist(rec_vals, bins=12, color='#E74C3C', edgecolor='black', alpha=0.8)
    ax_b.axvline(rec_vals.mean(), color='navy', linewidth=2, label=f'Mean={rec_vals.mean():.1f}s')
    ax_b.set_xlabel('Recovery_s (seconds)', fontsize=10, fontweight='bold')
    ax_b.set_ylabel('Count', fontsize=10, fontweight='bold')
    ax_b.set_title('Objective Recovery Time', fontsize=11, fontweight='bold')
    ax_b.legend(fontsize=9)
    ax_b.grid(alpha=0.3)
    bold_ax(ax_b, 9)

    # Panel C: Overall distracted rating vs BAPS
    ax_c = fig.add_subplot(gs[0, 2])
    df_oc = mdf[['overall_distract', 'baps_overall']].dropna()
    if len(df_oc) > 3:
        for rating in sorted(df_oc['overall_distract'].unique()):
            subset = df_oc[df_oc['overall_distract'] == rating]['baps_overall']
            ax_c.scatter([rating]*len(subset), subset, alpha=0.5, s=40, zorder=3)
        m, b_c = np.polyfit(df_oc['overall_distract'], df_oc['baps_overall'], 1)
        xr = np.linspace(1, 5, 50)
        ax_c.plot(xr, m*xr + b_c, '--', color='black', linewidth=2)
        r_c, p_c = safe_corr(df_oc['overall_distract'].values, df_oc['baps_overall'].values)
        ax_c.set_title(f'Overall Distracted (Q) vs BAPS\nr={r_c:+.3f}', fontsize=11, fontweight='bold')
    ax_c.set_xlabel('Overall Distraction Rating (1–5)', fontsize=10, fontweight='bold')
    ax_c.set_ylabel('BAPS_s (%)', fontsize=10, fontweight='bold')
    ax_c.axhline(100, color='gray', linestyle=':', alpha=0.5)
    ax_c.set_xticks([1,2,3,4,5])
    ax_c.grid(alpha=0.3)
    bold_ax(ax_c, 9)

    # Panel D: Self-report recovery vs objective recovery
    ax_d = fig.add_subplot(gs[1, 0])
    df_rd = mdf[['quick_recover', 'recovery_s']].dropna()
    if len(df_rd) > 3:
        ax_d.scatter(df_rd['quick_recover'], df_rd['recovery_s'], alpha=0.6, s=50, color='#9B59B6')
        m, b_d = np.polyfit(df_rd['quick_recover'], df_rd['recovery_s'], 1)
        xr = np.linspace(1, 5, 50)
        ax_d.plot(xr, m*xr + b_d, '--', color='black', linewidth=2)
        r_d, p_d = safe_corr(df_rd['quick_recover'].values, df_rd['recovery_s'].values)
        ax_d.set_title(f'Recovery Self-Report vs Recovery_s\nr={r_d:+.3f}', fontsize=11, fontweight='bold')
    ax_d.set_xlabel('Quick Recovery Self-Report (1–5)', fontsize=10, fontweight='bold')
    ax_d.set_ylabel('Recovery_s (objective, seconds)', fontsize=10, fontweight='bold')
    ax_d.invert_xaxis()
    ax_d.grid(alpha=0.3)
    bold_ax(ax_d, 9)

    # Panel E: Stressed vs ΔHR vs Set 1
    ax_e = fig.add_subplot(gs[1, 1])
    if 'hr_elev_s2' in mdf.columns:
        df_hr2 = mdf[['stressed', 'hr_elev_s2']].dropna()
        if len(df_hr2) > 3:
            ax_e.scatter(df_hr2['stressed'], df_hr2['hr_elev_s2'], alpha=0.6, s=50, color='#E74C3C')
            m, b_e = np.polyfit(df_hr2['stressed'], df_hr2['hr_elev_s2'], 1)
            xr = np.linspace(1, 5, 50)
            ax_e.plot(xr, m*xr + b_e, '--', color='black', linewidth=2)
            r_e, _ = safe_corr(df_hr2['stressed'].values, df_hr2['hr_elev_s2'].values)
            ax_e.set_title(f'Stressed (Q) vs ΔHR vs Set 1 (Set 2)\nr={r_e:+.3f}', fontsize=11, fontweight='bold')
    ax_e.set_xlabel('Felt Stressed (1–5)', fontsize=10, fontweight='bold')
    ax_e.set_ylabel('ΔHR vs Set 1 — Set 2 (bpm)', fontsize=10, fontweight='bold')
    ax_e.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax_e.grid(alpha=0.3)
    bold_ax(ax_e, 9)

    # Panel F: BAPS_s vs accuracy
    ax_f = fig.add_subplot(gs[1, 2])
    df_ac = mdf[['accuracy_overall', 'baps_overall']].dropna()
    if len(df_ac) > 3:
        ax_f.scatter(df_ac['accuracy_overall'], df_ac['baps_overall'], alpha=0.6, s=50, color='#27AE60')
        r_f, _ = safe_corr(df_ac['accuracy_overall'].values, df_ac['baps_overall'].values)
        ax_f.set_title(f'Accuracy vs BAPS_s\nr={r_f:+.3f}', fontsize=11, fontweight='bold')
    ax_f.set_xlabel('Accuracy % (distraction conditions)', fontsize=10, fontweight='bold')
    ax_f.set_ylabel('BAPS_s Overall (%)', fontsize=10, fontweight='bold')
    ax_f.grid(alpha=0.3)
    bold_ax(ax_f, 9)

    fig.suptitle('Subjective vs Objective Analysis — Summary Dashboard', fontsize=14, fontweight='bold')
    savefig(fig, 'plot10_summary_dashboard.png')

    print(f"\n✓ {len(saved)} plots saved to: {PLOTS_DIR}/")
    for s in saved:
        print(f"   {s}")
    return PLOTS_DIR


# Run this cell to execute the script
generate_report()
print("\nGenerating visualisations...")
generate_plots()
